In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression

# =========================================================
# Load data
# =========================================================
data = pd.read_csv("C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv")
screen = pd.read_csv("C:/Users/aiden/School/6380/in-silico-drug-discovery/data/list_of_compounds_for_computational_screening.csv")
research_preds = pd.read_csv("ResearchPaperPreds.csv")

# =========================================================
# Prepare training data
# =========================================================
labeled = data[data["senolytic"].isin([0, 1])].copy()

X_train = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y_train = labeled["senolytic"].values

# Align columns (important!)
X_screen = screen[X_train.columns].copy()

# =========================================================
# Variance filter (fit on train)
# =========================================================
variances = np.var(X_train.values, axis=0)
keep_var = variances > 1e-6

X_train = X_train.iloc[:, keep_var]
X_screen = X_screen.iloc[:, keep_var]

# =========================================================
# Scaling
# =========================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_screen_scaled = scaler.transform(X_screen)

# =========================================================
# PCA (for distance)
# =========================================================
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_screen_pca = pca.transform(X_screen_scaled)

# =========================================================
# KNN fit
# =========================================================
k = 50
knn = NearestNeighbors(n_neighbors=k, metric="cosine")
knn.fit(X_train_pca)

# =========================================================
# Prediction loop
# =========================================================
probs = []

for i in range(X_screen_pca.shape[0]):

    x_test_scaled = X_screen_scaled[i].reshape(1, -1)
    x_test_pca = X_screen_pca[i].reshape(1, -1)

    distances, indices = knn.kneighbors(x_test_pca)

    neighbor_idx = indices[0]
    dists = distances[0]

    X_neighbors = X_train_scaled[neighbor_idx]
    y_neighbors = y_train[neighbor_idx]

    pos_count = np.sum(y_neighbors == 1)
    neg_count = np.sum(y_neighbors == 0)

    # ----------------------------------------
    # Distance weights
    # ----------------------------------------
    sigma = np.mean(dists) + 1e-8
    weights = np.exp(-(dists**2) / (2 * sigma**2))

    if np.sum(weights) == 0:
        weights = np.ones_like(weights)

    # ----------------------------------------
    # Fallback logic
    # ----------------------------------------
    if pos_count == 1 and neg_count > 2:
        prob = 0.02
    elif neg_count == 1 and pos_count > 2:
        prob = 0.98
    elif pos_count == 0:
        prob = 0.001
    elif neg_count == 0:
        prob = 0.999
    else:
        ratio = neg_count / (pos_count + 1e-8)

        if ratio >= 1:
            w_pos = min(.65 * ratio, 4)
            w_neg = 1.0
        else:
            inv_ratio = 1 / (ratio + 1e-8)
            w_neg = min(inv_ratio, 1.1)
            w_pos = 1.0

        scale = (w_pos + w_neg) / 2
        w_pos /= scale
        w_neg /= scale

        sample_weights = weights.copy()
        sample_weights[y_neighbors == 1] *= w_pos
        sample_weights[y_neighbors == 0] *= w_neg

        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=5000
        )

        model.fit(X_neighbors, y_neighbors, sample_weight=sample_weights)

        prob = model.predict_proba(x_test_scaled)[0, 1]

        # confidence adjustment
        neighbor_preds = model.predict(X_neighbors)

        confidence = np.average(
            (neighbor_preds == y_neighbors).astype(float),
            weights=weights
        )

        if confidence > 0.95 and 0.35 < prob < 0.51:
            prob = 0.51

    probs.append(prob)

# =========================================================
# Attach predictions
# =========================================================
screen["prediction_prob"] = probs

# =========================================================
# Top 30 predictions
# =========================================================
top30 = screen.sort_values("prediction_prob", ascending=False).head(30)

print("\nTOP 30 PREDICTIONS:")
print(top30[["Name", "prediction_prob"]])

# =========================================================
# Cross-reference with research paper preds
# =========================================================
# Normalize names for matching
top30_names = set(top30["Name"].str.lower())
research_names = set(research_preds["Name"].str.lower())

overlap = top30_names.intersection(research_names)

overlap_df = top30[top30["Name"].str.lower().isin(overlap)]

print("\nOVERLAP WITH RESEARCH PAPER:")
print(overlap_df[["Name", "prediction_prob"]])

# =========================================================
# Save results
# =========================================================
top30.to_csv("Top30_Model_Preds.csv", index=False)
overlap_df.to_csv("Overlap_With_Research.csv", index=False)


TOP 30 PREDICTIONS:
                     Name  prediction_prob
2601     Milbemycin Oxime         0.942695
3590           Selamectin         0.836745
423            Aprepitant         0.742201
2692           Moxidectin         0.726188
3767  Sulconazole Nitrate         0.714610
3078           Periplocin         0.660398
2475            Mangostin         0.660258
2688              Morusin         0.627060
1754      gamma-Mangostin         0.605725
3471         Rostafuroxin         0.533705
3339           Quinestrol         0.533272
3417        Ridaforolimus         0.533241
2818          Nisoldipine         0.522691
3083            PF 573228         0.522378
1292            Depofemin         0.520035
1410           Doramectin         0.512727
1669               FLLL32         0.510000
3280             Propofol         0.510000
3370            Rapamycin         0.510000
2044            Ilomastat         0.510000
4332          Zotarolimus         0.489852
1332      Dicyclomine HCl        

In [8]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression

# =========================================================
# Load data
# =========================================================
data = pd.read_csv("C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv")
screen = pd.read_csv("C:/Users/aiden/School/6380/in-silico-drug-discovery/data/list_of_compounds_for_computational_screening.csv")
research_preds = pd.read_csv("ResearchPaperPreds.csv")

# =========================================================
# Prepare training data
# =========================================================
labeled = data[data["senolytic"].isin([0, 1])].copy()

X_train = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y_train = labeled["senolytic"].values

# Align columns safely
X_screen = screen.reindex(columns=X_train.columns, fill_value=0)

# =========================================================
# Variance filter
# =========================================================
variances = np.var(X_train.values, axis=0)
keep_var = variances > 1e-6

X_train = X_train.iloc[:, keep_var]
X_screen = X_screen.iloc[:, keep_var]

# =========================================================
# Scaling
# =========================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_screen_scaled = scaler.transform(X_screen)

# =========================================================
# PCA (for distance only)
# =========================================================
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_screen_pca = pca.transform(X_screen_scaled)

# =========================================================
# KNN
# =========================================================
k = 50
knn = NearestNeighbors(n_neighbors=k, metric="cosine")
knn.fit(X_train_pca)

# =========================================================
# Prediction loop
# =========================================================
probs = []
cluster_strengths = []

for i in range(X_screen_pca.shape[0]):

    x_test_scaled = X_screen_scaled[i].reshape(1, -1)
    x_test_pca = X_screen_pca[i].reshape(1, -1)

    distances, indices = knn.kneighbors(x_test_pca)

    neighbor_idx = indices[0]
    dists = distances[0]

    X_neighbors = X_train_scaled[neighbor_idx]
    y_neighbors = y_train[neighbor_idx]

    pos_mask = (y_neighbors == 1)
    neg_mask = (y_neighbors == 0)

    pos_count = np.sum(pos_mask)
    neg_count = np.sum(neg_mask)

    # ----------------------------------------
    # Distance weights
    # ----------------------------------------
    sigma = np.mean(dists) + 1e-8
    weights = np.exp(-(dists**2) / (2 * sigma**2))

    if np.sum(weights) == 0:
        weights = np.ones_like(weights)

    # ----------------------------------------
    # Cluster strength (distance-weighted)
    # ----------------------------------------
    cluster_strength = np.sum(weights[pos_mask]) / np.sum(weights)
    cluster_strengths.append(cluster_strength)

    # ----------------------------------------
    # Fallback logic
    # ----------------------------------------
    if pos_count == 1 and neg_count > 2:
        prob = 0.02
    elif neg_count == 1 and pos_count > 2:
        prob = 0.98
    elif pos_count == 0:
        prob = 0.001
    elif neg_count == 0:
        prob = 0.999
    else:
        ratio = neg_count / (pos_count + 1e-8)

        if ratio >= 1:
            w_pos = min(0.65 * ratio, 4)
            w_neg = 1.0
        else:
            inv_ratio = 1 / (ratio + 1e-8)
            w_neg = min(inv_ratio, 1.1)
            w_pos = 1.0

        scale = (w_pos + w_neg) / 2
        w_pos /= scale
        w_neg /= scale

        sample_weights = weights.copy()
        sample_weights[pos_mask] *= w_pos
        sample_weights[neg_mask] *= w_neg

        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=5000
        )

        model.fit(X_neighbors, y_neighbors, sample_weight=sample_weights)

        prob = model.predict_proba(x_test_scaled)[0, 1]

        # ----------------------------------------
        # Confidence adjustment
        # ----------------------------------------
        neighbor_preds = model.predict(X_neighbors)

        confidence = np.average(
            (neighbor_preds == y_neighbors).astype(float),
            weights=weights
        )

        if confidence > 0.95 and 0.35 < prob < 0.51:
            prob = 0.51

    probs.append(prob)

# =========================================================
# Attach predictions
# =========================================================
screen["prediction_prob"] = probs
screen["cluster_strength"] = cluster_strengths
screen["pred"] = (screen["prediction_prob"] >= 0.5).astype(int)

# =========================================================
# Rank ONLY predicted positives
# =========================================================
predicted_positive = screen[screen["pred"] == 1].copy()

top21 = predicted_positive.sort_values(
    ["cluster_strength", "prediction_prob"],
    ascending=False
).head(21)

print("\nTOP 21 PREDICTIONS (CLUSTER-WEIGHTED):")
print(top21[["Name", "prediction_prob", "cluster_strength"]])

# =========================================================
# Cross-reference with research paper preds
# =========================================================
top21_names = set(top21["Name"].str.lower())
research_names = set(research_preds["Name"].str.lower())

overlap = top21_names.intersection(research_names)

overlap_df = top21[top21["Name"].str.lower().isin(overlap)]

print("\nOVERLAP WITH RESEARCH PAPER:")
print(overlap_df[["Name", "prediction_prob", "cluster_strength"]])

# =========================================================
# Save results
# =========================================================
top21.to_csv("Top21_Model_Preds.csv", index=False)
overlap_df.to_csv("Overlap_With_Research.csv", index=False)


TOP 21 PREDICTIONS (CLUSTER-WEIGHTED):
                     Name  prediction_prob  cluster_strength
3078           Periplocin         0.660354          0.239458
3590           Selamectin         0.836738          0.163627
2601     Milbemycin Oxime         0.942690          0.162325
2692           Moxidectin         0.726168          0.159723
2688              Morusin         0.627075          0.150375
2475            Mangostin         0.660240          0.145253
1410           Doramectin         0.512664          0.143743
1754      gamma-Mangostin         0.605713          0.140113
3370            Rapamycin         0.510000          0.138483
3417        Ridaforolimus         0.533305          0.137833
1292            Depofemin         0.520003          0.129419
3767  Sulconazole Nitrate         0.714602          0.097166
3083            PF 573228         0.522381          0.093257
3471         Rostafuroxin         0.533711          0.086844
3339           Quinestrol         0.533085   

In [12]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression

# =========================================================
# Load data
# =========================================================
data = pd.read_csv("C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv")
screen = pd.read_csv("C:/Users/aiden/School/6380/in-silico-drug-discovery/data/list_of_compounds_for_computational_screening.csv")
research_preds = pd.read_csv("ResearchPaperPreds.csv")

# =========================================================
# Prepare training data
# =========================================================
labeled = data[data["senolytic"].isin([0, 1])].copy()

X_train = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y_train = labeled["senolytic"].values

# Align screening data
X_screen = screen.reindex(columns=X_train.columns, fill_value=0)

# =========================================================
# Variance filter
# =========================================================
variances = np.var(X_train.values, axis=0)
keep_var = variances > 1e-6

X_train = X_train.iloc[:, keep_var]
X_screen = X_screen.iloc[:, keep_var]

# =========================================================
# Scaling
# =========================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_screen_scaled = scaler.transform(X_screen)

# =========================================================
# PCA (distance space)
# =========================================================
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_screen_pca = pca.transform(X_screen_scaled)

# =========================================================
# KNN
# =========================================================
k = 50
knn = NearestNeighbors(n_neighbors=k, metric="cosine")
knn.fit(X_train_pca)

# =========================================================
# Prediction loop
# =========================================================
probs = []

for i in range(X_screen_pca.shape[0]):

    x_test_scaled = X_screen_scaled[i].reshape(1, -1)
    x_test_pca = X_screen_pca[i].reshape(1, -1)

    distances, indices = knn.kneighbors(x_test_pca)

    neighbor_idx = indices[0]
    dists = distances[0]

    X_neighbors = X_train_scaled[neighbor_idx]
    y_neighbors = y_train[neighbor_idx]

    pos_count = np.sum(y_neighbors == 1)
    neg_count = np.sum(y_neighbors == 0)

    # ----------------------------------------
    # Distance weights
    # ----------------------------------------
    sigma = np.mean(dists) + 1e-8
    weights = np.exp(-(dists**2) / (2 * sigma**2))

    if np.sum(weights) == 0:
        weights = np.ones_like(weights)

    # ----------------------------------------
    # Fallback logic
    # ----------------------------------------
    if pos_count == 1 and neg_count > 2:
        prob = 0.02
    elif neg_count == 1 and pos_count > 2:
        prob = 0.98
    elif pos_count == 0:
        prob = 0.001
    elif neg_count == 0:
        prob = 0.999
    else:
        ratio = neg_count / (pos_count + 1e-8)

        if ratio >= 1:
            w_pos = min(.65 * ratio, 4)
            w_neg = 1.0
        else:
            inv_ratio = 1 / (ratio + 1e-8)
            w_neg = min(inv_ratio, 1.1)
            w_pos = 1.0

        scale = (w_pos + w_neg) / 2
        w_pos /= scale
        w_neg /= scale

        sample_weights = weights.copy()
        sample_weights[y_neighbors == 1] *= w_pos
        sample_weights[y_neighbors == 0] *= w_neg

        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=5000
        )

        model.fit(X_neighbors, y_neighbors, sample_weight=sample_weights)

        prob = model.predict_proba(x_test_scaled)[0, 1]

        neighbor_preds = model.predict(X_neighbors)

        confidence = np.average(
            (neighbor_preds == y_neighbors).astype(float),
            weights=weights
        )

        if confidence > 0.95 and 0.35 < prob < 0.51:
            prob = 0.51

    probs.append(prob)

# =========================================================
# Attach predictions
# =========================================================
screen["prediction_prob"] = probs

# =========================================================
# ALL predicted positives (optional threshold)
# =========================================================
threshold = 0.5
screen["pred"] = (screen["prediction_prob"] >= threshold).astype(int)

predicted_positive = screen[screen["pred"] == 1].copy()

print("\nTOTAL PREDICTED POSITIVES:", len(predicted_positive))

# =========================================================
# CROSS-REFERENCE WITH RESEARCH PAPER (ALL DATA)
# =========================================================

model_names = set(predicted_positive["Name"].str.lower())
research_names = set(research_preds["Name"].str.lower())
print(predicted_positive["Name"].str.lower())

overlap = model_names.intersection(research_names)

overlap_df = predicted_positive[
    predicted_positive["Name"].str.lower().isin(overlap)
]

# =========================================================
# RESULTS
# =========================================================
print("\n====================================")
print("OVERLAP RESULTS")
print("====================================")

print(f"Research compounds: {len(research_names)}")
print(f"Model predictions : {len(model_names)}")
print(f"Overlap count     : {len(overlap)}")

print("\nMATCHING COMPOUNDS:")
print(overlap_df[["Name", "prediction_prob"]])

# =========================================================
# METRICS
# =========================================================
recall_vs_paper = len(overlap) / len(research_names)
precision_vs_paper = len(overlap) / len(model_names)

print("\n====================================")
print("METRICS")
print("====================================")
print(f"Recall vs paper   : {recall_vs_paper:.4f}")
print(f"Precision vs model: {precision_vs_paper:.4f}")

# =========================================================
# SAVE
# =========================================================
overlap_df.to_csv("Overlap_KNN_Research.csv", index=False)


TOTAL PREDICTED POSITIVES: 20
423              aprepitant
1292              depofemin
1410             doramectin
1669                 flll32
1754        gamma-mangostin
2044              ilomastat
2475              mangostin
2601       milbemycin oxime
2688                morusin
2692             moxidectin
2818            nisoldipine
3078             periplocin
3083              pf 573228
3280               propofol
3339             quinestrol
3370              rapamycin
3417          ridaforolimus
3471           rostafuroxin
3590             selamectin
3767    sulconazole nitrate
Name: Name, dtype: object

OVERLAP RESULTS
Research compounds: 21
Model predictions : 20
Overlap count     : 4

MATCHING COMPOUNDS:
                 Name  prediction_prob
1754  gamma-Mangostin         0.605714
3078       Periplocin         0.660264
3370        Rapamycin         0.510000
3417    Ridaforolimus         0.533254

METRICS
Recall vs paper   : 0.1905
Precision vs model: 0.2000
